In [4]:
import os
from event_detection import clear_directory
from event_detection import find_events
from fault import Fault

def run_sim(sim_num):
    directory = r'Constant pore pressure/Alpha 1 with nuleation'

    #alpha, pll_spd, no_elements, damping, ppr, element size
    param_dict = {1: [1, 0.001, 200 / 0.0625, 0.02, 0, 0.0625],
                  2: [1, 0.001, 200 / 0.0625, 0.02, 0.5, 0.0625],
                  3: [1, 0.001, 200 / 0.0625, 0.02, 0.6, 0.0625],
                  4: [1, 0.001, 200 / 0.0625, 0.02, 0.75, 0.0625]}
    params = param_dict[sim_num]
    fault = Fault(params[2], alpha=params[0], pll_spd=params[1], damping_coef=params[3], block_spacing=params[5],
                  p_ratio=params[4])
    a = params[-1]
    alpha = params[0]
    ppr = params[-2]
    n = params[3]


    ts = 0.001;tn = 10350;chunk = 75

    for time in range(0, tn, chunk):

        if time == 0:
            fault.set_initial('Random', ODE='C_L_ODE_fwrd_continuum_damped_cnst_pp')
            np.save(fr'sim_run_temp_files/Sim_{sim_num}/end_state_{sim_num}', fault.state)
            clear_directory(fr'sim_run_temp_files/Sim_{sim_num}',
                            exceptions=fr'end_state_{sim_num}.npy')



        fault.state = np.load(fr'sim_run_temp_files/Sim_{sim_num}/end_state_{sim_num}.npy')
        if time in [10050, 10125,10200,10275]:
            x_vel = fault.simulate(time, time + chunk, ts)
            np.save(fr'for event plot/ppr_{ppr}/event_sample_SRF_{time}', x_vel)
            np.save(fr'sim_run_temp_files/Sim_{sim_num}/event_sample_SRF_{time}', x_vel[1])
            np.save(fr'sim_run_temp_files/Sim_{sim_num}/end_state_{sim_num}', fault.state)
            del x_vel
        else:
            vel = fault.simulate(time, time + chunk, ts)[1]
            np.save(fr'sim_run_temp_files/Sim_{sim_num}/end_state_{sim_num}', fault.state)
            np.save(fr'sim_run_temp_files/Sim_{sim_num}/event_sample_SRF_{time}', vel)
            del vel





        if (time + chunk) % (1 * chunk) == 0 and time != 0:


            #events = find_events(sim_num, t0=time, threshold_factor=0)

            #if os.path.isfile(directory + f'/events_SRF_PPcont_alpha={alpha}=_n={n}_a={a}_ppr={ppr}.npy'):
                #prv_events = np.load(directory + f'/events_SRF_PPcont_alpha={alpha}=_n={n}_a={a}_ppr={ppr}.npy')
                #new_events = np.vstack((prv_events, events))
                #np.save(directory + f'/events_SRF_PPcont_alpha={alpha}=_n={n}_a={a}_ppr={ppr}.npy', new_events)
            #else:
                #np.save(directory + f'/events_SRF_PPcont_alpha={alpha}=_n={n}_a={a}_ppr={ppr}.npy', events)

            clear_directory(f'sim_run_temp_files/Sim_{sim_num}',
                            exceptions=f'end_state_{sim_num}.npy')



    return True


In [2]:
params = [1, 0.001, 200 / 0.0625, 0.02, 0.5, 0.0625]
fault = Fault(params[2], l2=10, alpha=params[0], pll_spd=params[1], damping_coef=params[3], block_spacing=params[5], injection_pressure=10, normal_pressure=25, evl_PorePressure=True)
a = params[-1]
alpha = params[0]
ppr = params[-2]
n = params[3]

fault.set_initial('Random', ODE='C_L_ODE_fwrd_evl_pp')
vel = fault.simulate(0, 100, 0.001)[1]


ValueError: operands could not be broadcast together with shapes (3201,) (3200,) 

In [10]:

def run_sim_1(sim_num):
    directory = r'Constant pore pressure/Alpha 1 with nuleation'
    param_dict = {1: [1, 0.001, 200 / 0.0625, 0.02, 0.5, 0.0625],
                  2: [1, 0.001, 200 / 0.0625, 0.02, 0.95, 0.0625],
                  3: [1, 0.001, 200 / 0.0625, 0.02, 0.975, 0.0625],
                  4: [1, 0.001, 200 / 0.0625, 0.02, 1, 0.0625]}
    params = param_dict[sim_num]
    fault = Fault(params[2], alpha=params[0], pll_spd=params[1], damping_coef=params[3], block_spacing=params[5],
                  p_ratio=params[4])

    a = params[-1]
    alpha = params[0]
    ppr = params[-2]
    n = params[3]

    if params[-1] == 0.0625:
        ts = 0.001;tn = 50000;chunk = 75

    for time in range(0, tn, chunk):

        if time == 0:
            fault.set_initial('Random', ODE='C_L_ODE_fwrd_continuum_damped_cnst_pp')
            #np.save(fr'sim_run_temp_files/Sim_{sim_num}/end_state_{sim_num}', fault.state)
            #clear_directory(fr'sim_run_temp_files/Sim_{sim_num}',
                            #exceptions=fr'end_state_{sim_num}.npy')

        fault.state = np.load(fr'sim_run_temp_files/Sim_{sim_num}/end_state_{sim_num}.npy')
        vel = fault.simulate(time, time + chunk, ts)[1]
        np.save(fr'sim_run_temp_files/Sim_{sim_num}/end_state_{sim_num}', fault.state)
        np.save(fr'sim_run_temp_files/Sim_{sim_num}/event_sample_SRF_{time}', vel)

        if (time + chunk) % (1 * chunk) == 0 and time != 0:
            print('Save occouring')

            events = find_events(sim_num, t0=time, threshold_factor=0)

            if os.path.isfile(directory + f'/events_SRF_PPcont_alpha={alpha}=_n={n}_a={a}_ppr={ppr}.npy'):
                prv_events = np.load(directory + f'/events_SRF_PPcont_alpha={alpha}=_n={n}_a={a}_ppr={ppr}.npy')
                new_events = np.vstack((prv_events, events))
                np.save(directory + f'/events_SRF_PPcont_alpha={alpha}=_n={n}_a={a}_ppr={ppr}.npy', new_events)
            else:
                np.save(directory + f'/events_SRF_PPcont_alpha={alpha}=_n={n}_a={a}_ppr={ppr}.npy', events)

            clear_directory(f'sim_run_temp_files/Sim_{sim_num}',
                            exceptions=f'end_state_{sim_num}.npy')

    return True


In [9]:
params = [3, 0.001, 200 / 0.0625, 0.02, 0.775, 0.0625]
fault = Fault(params[2], alpha=params[0], pll_spd=params[1], damping_coef=params[3], block_spacing=params[5],
                  p_ratio=params[4]); fault1 = Fault(params[2], alpha=params[0], pll_spd=params[1], damping_coef=params[3], block_spacing=params[5],
                  p_ratio=params[4]); fault2 = Fault(params[2], alpha=params[0], pll_spd=params[1], damping_coef=params[3], block_spacing=params[5],
                  p_ratio=params[4]); fault3 = Fault(params[2], alpha=params[0], pll_spd=params[1], damping_coef=params[3], block_spacing=params[5],
                  p_ratio=params[4])
a = params[-1]
alpha = params[0]
ppr = params[-2]
n = params[3]
if params[-1] == 0.0625:
        ts = 0.001;tn = 50000;chunk = 75
fault.set_initial('Random', ODE='C_L_ODE_fwrd_continuum_damped_cnst_pp')
fault1.set_initial('Random', ODE='C_L_ODE_fwrd_continuum_damped_cnst_pp')
fault2.set_initial('Random', ODE='C_L_ODE_fwrd_continuum_damped_cnst_pp')
fault3.set_initial('Random', ODE='C_L_ODE_fwrd_continuum_damped_cnst_pp')

for time in range(0, tn, chunk):
    vel = fault.simulate(time, time + chunk, ts)[1]
    vel = fault1.simulate(time, time + chunk, ts)[1]
    vel = fault2.simulate(time, time + chunk, ts)[1]
    vel = fault3.simulate(time, time + chunk, ts)[1]
    if time == 1050:
        End_notification(content='Got to same point as last time')

l2 not set, ensure you simulate with one of the continuum models
l2 not set, ensure you simulate with one of the continuum models
l2 not set, ensure you simulate with one of the continuum models
l2 not set, ensure you simulate with one of the continuum models
Email sent successfully!


KeyboardInterrupt: 

In [3]:

def run_sim_2(sim_num):
    directory = r'Constant pore pressure/Alpha 1 with nuleation'
    param_dict = {1: [1, 0.001, 200 / 0.0625, 0.02, 0.775, 0.0625],
                  2: [1, 0.001, 200 / 0.0625, 0.02, 0.825, 0.0625],
                  3: [1, 0.001, 200 / 0.0625, 0.02, 0.875, 0.0625],
                  4: [1, 0.001, 200 / 0.0625, 0.02, 0.85, 0.0625]}
    params = param_dict[sim_num]
    fault = Fault(params[2], alpha=params[0], pll_spd=params[1], damping_coef=params[3], block_spacing=params[5],
                  p_ratio=params[4])
    a = params[-1]
    alpha = params[0]
    ppr = params[-2]
    n = params[3]

    if params[-1] == 0.0625:
        ts = 0.001;tn = 50000;chunk = 75

    for time in range(0, tn, chunk):

        if time == 0:
            fault.set_initial('Random', ODE='C_L_ODE_fwrd_continuum_damped_cnst_pp')
            np.save(fr'sim_run_temp_files/Sim_{sim_num}/end_state_{sim_num}', fault.state)
            clear_directory(fr'sim_run_temp_files/Sim_{sim_num}',
                            exceptions=fr'end_state_{sim_num}.npy')

        fault.state = np.load(fr'sim_run_temp_files/Sim_{sim_num}/end_state_{sim_num}.npy')
        vel = fault.simulate(time, time + chunk, ts)[1]
        np.save(fr'sim_run_temp_files/Sim_{sim_num}/end_state_{sim_num}', fault.state)
        np.save(fr'sim_run_temp_files/Sim_{sim_num}/event_sample_SRF_{time}', vel)

        if (time + chunk) % (1 * chunk) == 0 and time != 0:
            print('Save occouring')

            events = find_events(sim_num, t0=time, threshold_factor=0)

            if os.path.isfile(directory + f'/events_SRF_PPcont_alpha={alpha}=_n={n}_a={a}_ppr={ppr}.npy'):
                prv_events = np.load(directory + f'/events_SRF_PPcont_alpha={alpha}=_n={n}_a={a}_ppr={ppr}.npy')
                new_events = np.vstack((prv_events, events))
                np.save(directory + f'/events_SRF_PPcont_alpha={alpha}=_n={n}_a={a}_ppr={ppr}.npy', new_events)
            else:
                np.save(directory + f'/events_SRF_PPcont_alpha={alpha}=_n={n}_a={a}_ppr={ppr}.npy', events)

            clear_directory(f'sim_run_temp_files/Sim_{sim_num}',
                            exceptions=f'end_state_{sim_num}.npy')

    return True


In [ ]:

def run_sim_3(sim_num):
    directory = r'Constant pore pressure/Alpha 3 with nuleation'
    param_dict = {1: [3, 0.001, 200 / 0.0625, 0.02, 0.825, 0.0625],
                  2: [3, 0.001, 200 / 0.0625, 0.02, 0.875, 0.0625],
                  3: [3, 0.001, 200 / 0.0625, 0.02, 0.775, 0.0625],
                  4: [3, 0.001, 200 / 0.0625, 0.02, 0.85, 0.0625]}
    params = param_dict[sim_num]
    fault = Fault(params[2], alpha=params[0], pll_spd=params[1], damping_coef=params[3], block_spacing=params[5],
                  p_ratio=params[4])
    a = params[-1]
    alpha = params[0]
    ppr = params[-2]
    n = params[3]

    if params[-1] == 0.0625:
        ts = 0.001;tn = 50000;chunk = 75

    for time in range(0, tn, chunk):

        if time == 0:
            fault.set_initial('Random', ODE='C_L_ODE_fwrd_continuum_damped_cnst_pp')
            np.save(fr'sim_run_temp_files/Sim_{sim_num}/end_state_{sim_num}', fault.state)
            clear_directory(fr'sim_run_temp_files/Sim_{sim_num}',
                            exceptions=fr'end_state_{sim_num}.npy')

        fault.state = np.load(fr'sim_run_temp_files/Sim_{sim_num}/end_state_{sim_num}.npy')
        vel = fault.simulate(time, time + chunk, ts)[1]
        np.save(fr'sim_run_temp_files/Sim_{sim_num}/end_state_{sim_num}', fault.state)
        np.save(fr'sim_run_temp_files/Sim_{sim_num}/event_sample_SRF_{time}', vel)

        if (time + chunk) % (1 * chunk) == 0 and time != 0:
            print('Save occouring')

            events = find_events(sim_num, t0=time, threshold_factor=0)

            if os.path.isfile(directory + f'/events_SRF_PPcont_alpha={alpha}=_n={n}_a={a}_ppr={ppr}.npy'):
                prv_events = np.load(directory + f'/events_SRF_PPcont_alpha={alpha}=_n={n}_a={a}_ppr={ppr}.npy')
                new_events = np.vstack((prv_events, events))
                np.save(directory + f'/events_SRF_PPcont_alpha={alpha}=_n={n}_a={a}_ppr={ppr}.npy', new_events)
            else:
                np.save(directory + f'/events_SRF_PPcont_alpha={alpha}=_n={n}_a={a}_ppr={ppr}.npy', events)

            clear_directory(f'sim_run_temp_files/Sim_{sim_num}',
                            exceptions=f'end_state_{sim_num}.npy')

    return True


In [ ]:


def run_sim_4(sim_num):
    directory = r'/Users/flemints/DataspellProjects/VSRP-Code/event_data_SRF/Original_friction__clamped_velocity/Pore Pressure in continuum limit/Constant pore pressure/Alpha 3 with nuleation'
    param_dict = {1: [1, 0.001, 200 / 0.0625, 0.02, 0.725, 0.0625],
                  2: [1, 0.001, 200 / 0.0625, 0.02, 0.75, 0.0625],
                  3: [3, 0.001, 200 / 0.0625, 0.02, 0.725, 0.0625],
                  4: [3, 0.001, 200 / 0.0625, 0.02, 0.75, 0.0625]}
    params = param_dict[sim_num]
    fault = Fault(params[2], alpha=params[0], pll_spd=params[1], damping_coef=params[3], block_spacing=params[5],
                  p_ratio=params[4])
    a = params[-1]
    alpha = params[0]
    ppr = params[-2]
    n = params[3]

    if params[-1] == 0.0625:
        ts = 0.001;tn = 50000;chunk = 75

    for time in range(0, tn, chunk):

        if time == 0:
            fault.set_initial('Random', ODE='C_L_ODE_fwrd_continuum_damped_cnst_pp')
            np.save(fr'sim_run_temp_files/Sim_{sim_num}/end_state_{sim_num}', fault.state)
            clear_directory(fr'sim_run_temp_files/Sim_{sim_num}',
                            exceptions=fr'end_state_{sim_num}.npy')

        fault.state = np.load(fr'sim_run_temp_files/Sim_{sim_num}/end_state_{sim_num}.npy')
        vel = fault.simulate(time, time + chunk, ts)[1]
        np.save(fr'sim_run_temp_files/Sim_{sim_num}/end_state_{sim_num}', fault.state)
        np.save(fr'sim_run_temp_files/Sim_{sim_num}/event_sample_SRF_{time}', vel)

        if (time + chunk) % (1 * chunk) == 0 and time != 0:
            print('Save occouring')

            events = find_events(sim_num, t0=time, threshold_factor=0)

            if os.path.isfile(directory + f'/events_SRF_PPcont_alpha={alpha}=_n={n}_a={a}_ppr={ppr}.npy'):
                prv_events = np.load(directory + f'/events_SRF_PPcont_alpha={alpha}=_n={n}_a={a}_ppr={ppr}.npy')
                new_events = np.vstack((prv_events, events))
                np.save(directory + f'/events_SRF_PPcont_alpha={alpha}=_n={n}_a={a}_ppr={ppr}.npy', new_events)
            else:
                np.save(directory + f'/events_SRF_PPcont_alpha={alpha}=_n={n}_a={a}_ppr={ppr}.npy', events)

            clear_directory(f'sim_run_temp_files/Sim_{sim_num}',
                            exceptions=f'end_state_{sim_num}.npy')

    return True


In [6]:
import traceback
from joblib import Parallel, delayed

sim_num = [1,2]

try:

    Parallel(n_jobs=-1, backend="loky")(
        delayed(run_sim)(n) for n in sim_num)

except Exception as e:
    print(e)
    err_msg = traceback.format_exc()
    traceback.print_exc()



Email sent successfully!


In [11]:
import traceback
from fault import Fault
from joblib import Parallel, delayed
import numpy as np
sim_num = [1, 2, 3]

try:

    Parallel(n_jobs=-1, backend="loky")(
        delayed(run_sim_1)(n) for n in sim_num)

except Exception as e:
    print(e)
    err_msg = traceback.format_exc()
    traceback.print_exc()




    

Email sent successfully!


In [ ]:
sim_num = [3, 4]
try:

    Parallel(n_jobs=-1, backend="loky")(
        delayed(run_sim_1)(n) for n in sim_num)
except Exception as e:
    print(e)

In [ ]:

sim_num = [3,4]
try:

    Parallel(n_jobs=-1, backend="loky")(
        delayed(run_sim_2)(n) for n in sim_num)

except Exception as e:
    print(e)
    err_msg = traceback.format_exc()
    traceback.print_exc()


In [ ]:
sim_num = [1, 2, 3, 4]
try:

    Parallel(n_jobs=-1, backend="loky")(
        delayed(run_sim_3)(n) for n in sim_num)

except Exception as e:
    print(e)

In [ ]:
sim_num = [1, 2, 3, 4]
try:

    Parallel(n_jobs=-1, backend="loky")(
        delayed(run_sim_4)(n) for n in sim_num)

except Exception as e:
    print(e)